# Week 1: Light Exercises (~3h)

Scratch work: `dataclasses`, slicing, comprehensions, dict-as-switch patterns.

**Core exercises** (~2h): 1a, 2a, 3a, 4a+4b  
**Bonus** (~1h): 1b, 2b, 3b, 3c, 4c  

Don't polish anything — feel the idioms, not ship code.

---
## 1. Dataclasses

### Exercise 1a: Batch record model

Create a `BatchRecord` dataclass with fields: `batch_id: str`, `product: str`, `yield_pct: float`, `date: date`, `passed_qc: bool`.

- Add a `__post_init__` that raises `ValueError` if `yield_pct` is outside 0–100.
- Instantiate a few, throw them in a list, sort by yield.
- Then try: make a `frozen=True` version and see what breaks when you try to mutate it.
- Compare with how you'd do this in R (probably a tibble row or a named list) — notice how much structure you get for free.

In [31]:
from dataclasses import dataclass, asdict, field
from datetime import date

# Your code here

@dataclass(frozen=True)
class BatchRecord:
    batch_id: str
    product: str
    yield_pct: float
    date: date
    passed_qc: bool
    def __post_init__(self):
       if not (0 <= self.yield_pct <= 100):
           raise ValueError(f"yield_pct must be 0-100, got {self.yield_pct}")


In [43]:
b0 = BatchRecord("BATCH-001", "CellTherapyX", 95.3, date(2026, 4, 20), True)
b1 = BatchRecord("BATCH-002", "CellTherapyX", 83.0, date(2026, 4, 21), False)
b2 = BatchRecord("BATCH-003", "CellTherapyX", 99.0, date(2026, 4, 22), True)
b3 = BatchRecord("BATCH-004", "CellTherapyX", 11, date(2026, 4, 23), False)
ppq = [b0, b1, b2, b3]
sorted(ppq, key = lambda x:  x.passed_qc)

b3.yield_pct = 13

TypeError: BatchRecord.__init__() takes 3 positional arguments but 6 were given

### Exercise 1b: Nested dataclasses (bonus)

Create a `CQA` (critical quality attribute) dataclass with `name`, `value`, `unit`, `spec_lower`, `spec_upper`.

- Add a `@property` called `in_spec` that returns a bool.
- Create a `BatchReport` dataclass that holds a `BatchRecord` and a `list[CQA]`.
- Add a method `out_of_spec_attributes` that returns only the failing CQAs.
- Play with `asdict()` — notice how it recursively converts nested dataclasses to dicts.

In [40]:
# Your code here
@dataclass
class CQA:
    name: str
    value: float
    unit: str
    spec_lower: float
    spec_upper: float

    @property
    def in_spec(self):
        return self.spec_lower < self.value < self.spec_upper 

@dataclass
class BatchRecord:
    BatchRecord: BatchRecord
    cqas: CQA

In [42]:
p1=CQA("power", 0.5, "W", 1, 10)
p2=CQA("volume", 20, "L", 10, 100)
cqas = [p1, p2]
print(p1.in_spec)

b11 = BatchRecord(b1, cqas)

False


---
## 2. Slicing

### Exercise 2a: Named slices for readability

Simulate fixed-width manufacturing log lines like:
```
"BATCH-0042  2026-04-20  FILLING   98.3  PASS"
```

- Define named slices (`BATCH = slice(0, 10)`, `DATE = slice(12, 22)`, etc.)
- Parse 5–10 lines using only slicing.
- Compare the readability of `line[BATCH]` vs positional magic numbers.

In [50]:
# Sample log lines — adjust column widths to match your slices
log_lines = [
    "BATCH-0042  2026-04-20  FILLING   98.3  PASS",
    "BATCH-0043  2026-04-20  FILLING   95.1  PASS",
    "BATCH-0044  2026-04-21  COATING   87.6  FAIL",
    "BATCH-0045  2026-04-21  FILLING   99.2  PASS",
    "BATCH-0046  2026-04-22  COATING   91.0  PASS",
]

# Define named slices
BATCH = slice(0, 10)
DATE = slice(12, 22)
STEP = slice(24,31)
YIELD = slice(33, 38)
QC = slice(40, 45)
# Parse and print
log_lines[3][QC]

'PASS'

### Exercise 2b: Stride tricks (bonus)

Given a list of hourly sensor readings (just fake 48 values), use slicing to extract:
- Every other reading (downsampled)
- The last 12 readings reversed
- Readings from index 6 to 18 in steps of 3

Then try `readings[::-1]` vs `list(reversed(readings))` — think about when each is appropriate.

In [5]:
import random
random.seed(42)
readings = [round(random.gauss(37.0, 0.5), 2) for _ in range(48)]

# Your slicing here


---
## 3. Comprehensions

### Exercise 3a: Dict and set comprehensions

Using the list of `BatchRecord` dataclasses from exercise 1, create:
- A dict mapping `batch_id → yield_pct` (dict comprehension)
- A set of all unique products (set comprehension)
- A dict mapping `product → [list of batch_ids]` — try it with `defaultdict` too and compare

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, asdict, field
from datetime import date

# Your code here

@dataclass()
class BatchRecord:
    batch_id: str
    product: str
    yield_pct: float
    date: date
    passed_qc: bool
    def __post_init__(self):
       if not (0 <= self.yield_pct <= 100):
           raise ValueError(f"yield_pct must be 0-100, got {self.yield_pct}")

# Reuse your batches list from exercise 1a
b0 = BatchRecord("BATCH-001", "CellTherapyX", 95.3, date(2026, 4, 20), True)
b1 = BatchRecord("BATCH-002", "CellTherapyX", 83.0, date(2026, 4, 21), False)
b2 = BatchRecord("BATCH-003", "CellTherapyX", 99.0, date(2026, 4, 22), True)
b3 = BatchRecord("BATCH-004", "CellTherapyX", 11, date(2026, 4, 23), False)
ppq = [b0, b1, b2, b3]

# Dict comprehension: batch_id -> yield_pct
print({x.batch_id: x.yield_pct for x in ppq})

# Set comprehension: unique products
print(set([x.product for x in ppq]))

# product -> [batch_ids] — try both ways



{'BATCH-001': 95.3, 'BATCH-002': 83.0, 'BATCH-003': 99.0, 'BATCH-004': 11}
{'CellTherapyX'}


### Exercise 3b: Nested comprehension — the matrix (bonus)

- Create a 5×5 identity matrix as a list of lists using a nested comprehension.
- Flatten it back to 1D with another comprehension.
- Note: loop order reads left-to-right (outer → inner). This trips people up.

In [7]:
# Your code here


### Exercise 3c: Generator expression vs list comprehension (bonus)

- Write both a list comprehension and a generator expression filtering batches to `passed_qc == True`.
- Print `type()` of each.
- Consume the generator with `list()`, then try to consume it again — observe exhaustion.
- Key difference from R: everything in R is eagerly evaluated.

In [8]:
# Your code here


---
## 4. Dict-as-switch

### Exercise 4a: Status mapper

Write a function that takes a batch status string (`"pending"`, `"in_progress"`, `"released"`, `"rejected"`) and returns a formatted emoji + label (e.g., `"⏳ Pending"`).

- Implement first with `if/elif`.
- Rewrite as a dict lookup with `.get()` for the default case.
- Notice how much cleaner the dict version is.

In [9]:
# if/elif version


# dict version


# Test both


### Exercise 4b: Dispatch table

Create a dict mapping string keys to functions:
```python
operations = {
    "mean": statistics.mean,
    "median": statistics.median,
    "stdev": statistics.stdev,
    "range": lambda x: max(x) - min(x),
}
```

Write `summarize(data: list[float], ops: list[str]) → dict[str, float]` that looks up each operation and applies it. Handle unknown ops gracefully.

This pattern replaces R's `switch()` and is everywhere in Python codebases.

In [10]:
import statistics

# Your code here


### Exercise 4c: Registry pattern (stretch/bonus)

Write a decorator `@register("name")` that auto-adds functions to a registry dict. This is how plugin systems work in Python — you'll see it again with Flask/FastAPI route decorators.

In [11]:
# Your code here


---
## Notes

Jot down anything that felt un-R-like or surprising:

*Your notes here*